In [0]:
%sql
SELECT
    current_catalog() AS current_catalog,
    current_schema() AS current_schema;

current_catalog,current_schema
workspace,default


In [0]:
%sql
SHOW CATALOGS;

catalog
samples
system
workspace


In [0]:

%sql
CREATE SCHEMA IF NOT EXISTS workspace.token_risk_bronze
COMMENT 'Source-preserving SDB blockchain transfer data';

CREATE SCHEMA IF NOT EXISTS workspace.token_risk_silver
COMMENT 'Validated, typed and normalized SDB transfer data';

CREATE SCHEMA IF NOT EXISTS workspace.token_risk_gold
COMMENT 'Buyer-oriented SDB risk intelligence aggregates';

In [0]:
%sql
SHOW SCHEMAS IN workspace;

databaseName
default
information_schema
token_risk_bronze
token_risk_gold
token_risk_silver


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS
    workspace.token_risk_bronze.landing_files
COMMENT 'Governed landing area for validated SDB source files';

In [0]:
%sql
SHOW VOLUMES IN workspace.token_risk_bronze;

database,volume_name
token_risk_bronze,landing_files


In [0]:
%python
import json

RAW_FILE_PATH = (
    "/Volumes/workspace/token_risk_bronze/"
    "landing_files/sdb_listing_window.jsonl"
)

REPORT_FILE_PATH = (
    "/Volumes/workspace/token_risk_bronze/"
    "landing_files/sdb_listing_window_validation.json"
)

with open(REPORT_FILE_PATH, "r", encoding="utf-8") as report_file:
    validation_report = json.load(report_file)

expected_sha256 = validation_report["source"]["sha256"]

print("Expected SHA-256:", expected_sha256)

Expected SHA-256: 2582bdd1f8e2a7850ff41ebc439581578f79aea83af25114fce6d0d79868b8e9


In [0]:
%python
import hashlib

digest = hashlib.sha256()
with open(RAW_FILE_PATH, "rb") as raw_file:
    while True:
        chunk = raw_file.read(1024 * 1024)

        if not chunk:
            break

        digest.update(chunk)
actual_sha256 = digest.hexdigest()

print("Actual SHA-256:  ", actual_sha256)
print("Checksum match:  ", actual_sha256 == expected_sha256)

Actual SHA-256:   2582bdd1f8e2a7850ff41ebc439581578f79aea83af25114fce6d0d79868b8e9
Checksum match:   True


In [0]:

%python
from pyspark.sql.types import StringType, StructField, StructType
raw_df = spark.read.json(RAW_FILE_PATH)
raw_df.printSchema()
raw_count = raw_df.count()
print("Raw Spark records:", raw_count)
expected_count = validation_report["counts"]["valid_json_records"]

if raw_count != expected_count:
    raise RuntimeError(
        f"Row-count mismatch: expected {expected_count}, "
        f"but Spark read {raw_count}."
    )
display(raw_df.limit(5))
RAW_FIELDS = [
    "blockHash",
    "blockNumber",
    "confirmations",
    "contractAddress",
    "cumulativeGasUsed",
    "from",
    "functionName",
    "gas",
    "gasPrice",
    "gasUsed",
    "hash",
    "input",
    "methodId",
    "nonce",
    "statusRep",
    "timeStamp",
    "to",
    "tokenDecimal",
    "tokenName",
    "tokenSymbol",
    "transactionIndex",
    "value",
]
bronze_schema = StructType([
    StructField(field_name, StringType(), True)
    for field_name in RAW_FIELDS
])
unexpected_fields = sorted(set(raw_df.columns) - set(RAW_FIELDS))
missing_fields = sorted(set(RAW_FIELDS) - set(raw_df.columns))

print("Unexpected fields:", unexpected_fields)
print("Missing fields:", missing_fields)
if unexpected_fields or missing_fields:
    raise RuntimeError(
        "The raw API schema does not match the expected Bronze schema."
    )
bronze_source_df = (
    spark.read
    .schema(bronze_schema)
    .option("mode", "FAILFAST")
    .json(RAW_FILE_PATH)
)
print("Explicit-schema records:", bronze_source_df.count())
bronze_source_df.printSchema()

root
 |-- blockHash: string (nullable = true)
 |-- blockNumber: string (nullable = true)
 |-- confirmations: string (nullable = true)
 |-- contractAddress: string (nullable = true)
 |-- cumulativeGasUsed: string (nullable = true)
 |-- from: string (nullable = true)
 |-- functionName: string (nullable = true)
 |-- gas: string (nullable = true)
 |-- gasPrice: string (nullable = true)
 |-- gasUsed: string (nullable = true)
 |-- hash: string (nullable = true)
 |-- input: string (nullable = true)
 |-- methodId: string (nullable = true)
 |-- nonce: string (nullable = true)
 |-- statusRep: string (nullable = true)
 |-- timeStamp: string (nullable = true)
 |-- to: string (nullable = true)
 |-- tokenDecimal: string (nullable = true)
 |-- tokenName: string (nullable = true)
 |-- tokenSymbol: string (nullable = true)
 |-- transactionIndex: string (nullable = true)
 |-- value: string (nullable = true)

Raw Spark records: 2656


blockHash,blockNumber,confirmations,contractAddress,cumulativeGasUsed,from,functionName,gas,gasPrice,gasUsed,hash,input,methodId,nonce,statusRep,timeStamp,to,tokenDecimal,tokenName,tokenSymbol,transactionIndex,value
0x9b7444f6d98bdd4d67855b7801295a13e880dfaa5759ace7c12083f0bacc28fb,78870787,13571966,0xd2d21ebc27dc39e188bf51fa28d3d09b93ab49c8,15473949,0x8e95308b721d3fa0066e3e02653d1709df4a8893,"execTransaction(address to, uint256 value, bytes data, uint8 operation, uint256 safeTxGas, uint256 baseGas, uint256 gasPrice, address gasToken, address refundReceiver, bytes signatures)",112804,292320124078,111584,0x3c62828bc161d23da0cfc0dc29b44d8c9b7d3e6a478dad3d31deca5114f68ffb,deprecated,0x6a761202,10,0,1762846459,0x1133209228922ed63279c309972cee654fb6d5fe,18,Spring Development Bank Token,SDB,94,6000000000000000000000000
0x3e34f1f2d7ac69d3d0d384d6cb567bc870438d5facd2fe4413dabf5a76ec38b8,78879749,13563004,0xd2d21ebc27dc39e188bf51fa28d3d09b93ab49c8,20556892,0x317c735bf712eb858446a13049f17fe492132101,"batchTransferERC20(address token, address[] _contributors, uint256[] _amounts)",108796,260639888850,85083,0xe22c4e72c8b225704a702a324d6197f3c92176c20c4bac500b2e9e5ab3bb84b0,deprecated,0x79f2447e,7,0,1762864383,0xa77cc444db535ea85eefcbac2dc6aa07cd445a70,18,Spring Development Bank Token,SDB,117,200000000000000000000
0x3e34f1f2d7ac69d3d0d384d6cb567bc870438d5facd2fe4413dabf5a76ec38b8,78879749,13563004,0xd2d21ebc27dc39e188bf51fa28d3d09b93ab49c8,20556892,0xa77cc444db535ea85eefcbac2dc6aa07cd445a70,"batchTransferERC20(address token, address[] _contributors, uint256[] _amounts)",108796,260639888850,85083,0xe22c4e72c8b225704a702a324d6197f3c92176c20c4bac500b2e9e5ab3bb84b0,deprecated,0x79f2447e,7,0,1762864383,0x3627ce35a466e86662b602e925c9dbfdba14f1a2,18,Spring Development Bank Token,SDB,117,200000000000000000000
0xb8b8c9bc437394ef4739651a9c2f9fd9b72cc7298fbace0345196f02fcc23008,78930769,13511984,0xd2d21ebc27dc39e188bf51fa28d3d09b93ab49c8,25273451,0x8e95308b721d3fa0066e3e02653d1709df4a8893,"execTransaction(address to, uint256 value, bytes data, uint8 operation, uint256 safeTxGas, uint256 baseGas, uint256 gasPrice, address gasToken, address refundReceiver, bytes signatures)",112816,153778764244,111596,0x36d69c80f49c3c20f4502ead7a984d5d0f449b0c21e1e81e309ca83311f5006f,deprecated,0x6a761202,11,0,1762966423,0x2e03695a934a3586b05fe2fba74543f7546b500d,18,Spring Development Bank Token,SDB,115,9000000000000000000000000
0x666a730611b985d19a726a5914bf14bafbef935b077c5ca1395141baa186555e,78959651,13483102,0xd2d21ebc27dc39e188bf51fa28d3d09b93ab49c8,24871434,0x317c735bf712eb858446a13049f17fe492132101,"batchTransferERC20(address token, address[] _contributors, uint256[] _amounts)",108807,308599700749,85092,0x9db6ab06f60b70966b335335dc32196f50c5af35e3ffb66efa27836aa9acaad5,deprecated,0x79f2447e,10,0,1763024188,0xa77cc444db535ea85eefcbac2dc6aa07cd445a70,18,Spring Development Bank Token,SDB,103,39600000000000000000000


Unexpected fields: []
Missing fields: []
Explicit-schema records: 2656
root
 |-- blockHash: string (nullable = true)
 |-- blockNumber: string (nullable = true)
 |-- confirmations: string (nullable = true)
 |-- contractAddress: string (nullable = true)
 |-- cumulativeGasUsed: string (nullable = true)
 |-- from: string (nullable = true)
 |-- functionName: string (nullable = true)
 |-- gas: string (nullable = true)
 |-- gasPrice: string (nullable = true)
 |-- gasUsed: string (nullable = true)
 |-- hash: string (nullable = true)
 |-- input: string (nullable = true)
 |-- methodId: string (nullable = true)
 |-- nonce: string (nullable = true)
 |-- statusRep: string (nullable = true)
 |-- timeStamp: string (nullable = true)
 |-- to: string (nullable = true)
 |-- tokenDecimal: string (nullable = true)
 |-- tokenName: string (nullable = true)
 |-- tokenSymbol: string (nullable = true)
 |-- transactionIndex: string (nullable = true)
 |-- value: string (nullable = true)



In [0]:
%python
import uuid
from pyspark.sql import functions as F

ingestion_run_id = str(uuid.uuid4())

source_metadata = validation_report["source"]
window_metadata = validation_report["requested_window"]
bronze_df = (
    bronze_source_df
    .withColumn("ingested_at_utc", F.current_timestamp())
    .withColumn("ingestion_run_id", F.lit(ingestion_run_id))
    .withColumn("source_file", F.lit(RAW_FILE_PATH))
    .withColumn("source_sha256", F.lit(actual_sha256))
    .withColumn(
        "source_chain_id",
        F.lit(int(source_metadata["chain_id"])),
    )
    .withColumn(
        "source_contract_address",
        F.lit(source_metadata["contract_address"]),
    )
    .withColumn(
        "research_start_block",
        F.lit(int(window_metadata["start_block"])),
    )
    .withColumn(
        "research_end_block",
        F.lit(int(window_metadata["end_block"])),
    )
    .withColumn(
        "source_validation_status",
        F.lit(validation_report["validation_status"]),
    )
)

BRONZE_TABLE = "workspace.token_risk_bronze.sdb_transfers"

(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)

In [0]:
%sql
SELECT COUNT(*) AS bronze_record_count
FROM workspace.token_risk_bronze.sdb_transfers;
DESCRIBE DETAIL workspace.token_risk_bronze.sdb_transfers;
SELECT
    source_validation_status,
    source_sha256,
    source_chain_id,
    research_start_block,
    research_end_block,
    COUNT(*) AS record_count
FROM workspace.token_risk_bronze.sdb_transfers
GROUP BY ALL;

source_validation_status,source_sha256,source_chain_id,research_start_block,research_end_block,record_count
PASS,2582bdd1f8e2a7850ff41ebc439581578f79aea83af25114fce6d0d79868b8e9,137,78813960,79289084,2656


In [0]:
%sql
SELECT COUNT(*) AS bronze_record_count
FROM workspace.token_risk_bronze.sdb_transfers;

DESCRIBE DETAIL workspace.token_risk_bronze.sdb_transfers;
SELECT
    source_validation_status,
    source_sha256,
    source_chain_id,
    research_start_block,
    research_end_block,
    COUNT(*) AS record_count
FROM workspace.token_risk_bronze.sdb_transfers
GROUP BY ALL;

source_validation_status,source_sha256,source_chain_id,research_start_block,research_end_block,record_count
PASS,2582bdd1f8e2a7850ff41ebc439581578f79aea83af25114fce6d0d79868b8e9,137,78813960,79289084,2656


In [0]:
%python
spark.conf.set("spark.sql.session.timeZone", "UTC")
from pyspark.sql import functions as F

bronze_table_df = spark.table(
    "workspace.token_risk_bronze.sdb_transfers"
)
silver_candidate_df = bronze_table_df.select(
    F.col("hash").alias("transaction_hash"),
    F.col("blockHash").alias("block_hash"),

    F.col("blockNumber")
        .cast("long")
        .alias("block_number"),

    F.col("transactionIndex")
        .cast("integer")
        .alias("transaction_index"),

    F.to_timestamp(
        F.from_unixtime(F.col("timeStamp").cast("long"))
    ).alias("event_timestamp_utc"),

    F.lower(F.col("from")).alias("from_address"),
    F.lower(F.col("to")).alias("to_address"),
    F.lower(F.col("contractAddress")).alias(
        "contract_address"
    ),

    F.col("value")
        .cast("decimal(38,0)")
        .alias("raw_token_value"),

    F.col("tokenDecimal")
        .cast("integer")
        .alias("token_decimals"),

    F.col("tokenName").alias("token_name"),
    F.upper(F.col("tokenSymbol")).alias("token_symbol"),

    "ingested_at_utc",
    "ingestion_run_id",
    "source_file",
    "source_sha256",
    "source_validation_status",
)
silver_candidate_df = silver_candidate_df.withColumn(
    "token_amount",
    (
        F.col("raw_token_value")
        / F.lit(1_000_000_000_000_000_000)
    ).cast("decimal(38,18)")
)
print("Silver candidate rows:", silver_candidate_df.count())
silver_candidate_df.printSchema()
display(
    silver_candidate_df.select(
        "transaction_hash",
        "event_timestamp_utc",
        "from_address",
        "to_address",
        "raw_token_value",
        "token_amount",
    ).limit(5)
)
CRITICAL_SILVER_COLUMNS = [
    "transaction_hash",
    "block_hash",
    "block_number",
    "transaction_index",
    "event_timestamp_utc",
    "from_address",
    "to_address",
    "contract_address",
    "raw_token_value",
    "token_decimals",
    "token_amount",
    "token_symbol",
]
null_count_row = silver_candidate_df.select([
    F.sum(
        F.col(column_name).isNull().cast("integer")
    ).alias(column_name)
    for column_name in CRITICAL_SILVER_COLUMNS
]).first()

null_counts = null_count_row.asDict()

for column_name, null_count in null_counts.items():
    print(f"{column_name}: {null_count}")
total_critical_nulls = sum(null_counts.values())

print("Total critical nulls:", total_critical_nulls)
if total_critical_nulls != 0:
    raise RuntimeError(
        "Silver transformation produced unexpected null values."
    )
expected_contract = (
    validation_report["source"]["contract_address"].lower()
)

expected_start_block = int(
    validation_report["requested_window"]["start_block"]
)

expected_end_block = int(
    validation_report["requested_window"]["end_block"]
)
quality_condition = (
    (F.col("contract_address") == expected_contract)
    & (F.col("token_symbol") == "SDB")
    & (F.col("token_decimals") == 18)
    & (F.col("raw_token_value") >= 0)
    & F.col("block_number").between(
        expected_start_block,
        expected_end_block,
    )
)
is_valid = F.coalesce(
    quality_condition,
    F.lit(False),
)
silver_valid_df = silver_candidate_df.filter(is_valid)
silver_rejected_df = silver_candidate_df.filter(~is_valid)

silver_valid_count = silver_valid_df.count()
silver_rejected_count = silver_rejected_df.count()

print("Silver valid rows:", silver_valid_count)
print("Silver rejected rows:", silver_rejected_count)

silver_valid_df = silver_valid_df.withColumn(
    "candidate_event_fingerprint",
    F.sha2(
        F.concat_ws(
            "|",
            F.col("transaction_hash"),
            F.col("block_number").cast("string"),
            F.col("transaction_index").cast("string"),
            F.col("contract_address"),
            F.col("from_address"),
            F.col("to_address"),
            F.col("raw_token_value").cast("string"),
        ),
        256,
    ),
)
SILVER_TABLE = "workspace.token_risk_silver.sdb_transfers"

(
    silver_valid_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

Silver candidate rows: 2656
root
 |-- transaction_hash: string (nullable = true)
 |-- block_hash: string (nullable = true)
 |-- block_number: long (nullable = true)
 |-- transaction_index: integer (nullable = true)
 |-- event_timestamp_utc: timestamp (nullable = true)
 |-- from_address: string (nullable = true)
 |-- to_address: string (nullable = true)
 |-- contract_address: string (nullable = true)
 |-- raw_token_value: decimal(38,0) (nullable = true)
 |-- token_decimals: integer (nullable = true)
 |-- token_name: string (nullable = true)
 |-- token_symbol: string (nullable = true)
 |-- ingested_at_utc: timestamp (nullable = true)
 |-- ingestion_run_id: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- source_sha256: string (nullable = true)
 |-- source_validation_status: string (nullable = true)
 |-- token_amount: decimal(38,18) (nullable = true)



transaction_hash,event_timestamp_utc,from_address,to_address,raw_token_value,token_amount
0x3c62828bc161d23da0cfc0dc29b44d8c9b7d3e6a478dad3d31deca5114f68ffb,2025-11-11T07:34:19.000Z,0x8e95308b721d3fa0066e3e02653d1709df4a8893,0x1133209228922ed63279c309972cee654fb6d5fe,6000000000000000000000000,6000000.000000000000000000
0xe22c4e72c8b225704a702a324d6197f3c92176c20c4bac500b2e9e5ab3bb84b0,2025-11-11T12:33:03.000Z,0x317c735bf712eb858446a13049f17fe492132101,0xa77cc444db535ea85eefcbac2dc6aa07cd445a70,200000000000000000000,200.000000000000000000
0xe22c4e72c8b225704a702a324d6197f3c92176c20c4bac500b2e9e5ab3bb84b0,2025-11-11T12:33:03.000Z,0xa77cc444db535ea85eefcbac2dc6aa07cd445a70,0x3627ce35a466e86662b602e925c9dbfdba14f1a2,200000000000000000000,200.000000000000000000
0x36d69c80f49c3c20f4502ead7a984d5d0f449b0c21e1e81e309ca83311f5006f,2025-11-12T16:53:43.000Z,0x8e95308b721d3fa0066e3e02653d1709df4a8893,0x2e03695a934a3586b05fe2fba74543f7546b500d,9000000000000000000000000,9000000.000000000000000000
0x9db6ab06f60b70966b335335dc32196f50c5af35e3ffb66efa27836aa9acaad5,2025-11-13T08:56:28.000Z,0x317c735bf712eb858446a13049f17fe492132101,0xa77cc444db535ea85eefcbac2dc6aa07cd445a70,39600000000000000000000,39600.000000000000000000


transaction_hash: 0
block_hash: 0
block_number: 0
transaction_index: 0
event_timestamp_utc: 0
from_address: 0
to_address: 0
contract_address: 0
raw_token_value: 0
token_decimals: 0
token_amount: 0
token_symbol: 0
Total critical nulls: 0
Silver valid rows: 2656
Silver rejected rows: 0


In [0]:
%sql
ALTER TABLE workspace.token_risk_silver.sdb_transfers
ALTER COLUMN candidate_event_fingerprint
COMMENT 'Deterministic signature from available tokentx fields; not a definitive event identifier because logIndex is unavailable';

In [0]:
%sql
CREATE OR REPLACE TABLE
    workspace.token_risk_gold.daily_transfer_activity
USING DELTA
COMMENT 'Daily SDB transfer activity during the MEXC listing research window'
AS
SELECT
    CAST(event_timestamp_utc AS DATE) AS event_date_utc,
    COUNT(*) AS transfer_count,
    SUM(token_amount) AS total_token_amount,
    AVG(token_amount) AS average_token_amount,
    MAX(token_amount) AS maximum_token_amount,
    COUNT(DISTINCT from_address) AS unique_senders,
    COUNT(DISTINCT to_address) AS unique_recipients
FROM workspace.token_risk_silver.sdb_transfers
GROUP BY CAST(event_timestamp_utc AS DATE);

SELECT *
FROM workspace.token_risk_gold.daily_transfer_activity
ORDER BY event_date_utc;

event_date_utc,transfer_count,total_token_amount,average_token_amount,maximum_token_amount,unique_senders,unique_recipients
2025-11-11,3,6000400.000000000000000000,2000133.3333333333333333333333,6000000.000000000000000000,3,3
2025-11-12,1,9000000.000000000000000000,9000000.0000000000000000000000,9000000.000000000000000000,1,1
2025-11-13,268,74793000.000000000000000000,279078.3582089552238805970149,20000000.000000000000000000,5,119
2025-11-14,914,195048407.100000000000000000,213400.8830415754923413566740,20000000.000000000000000000,193,323
2025-11-15,265,40863040.280000000000000000,154200.1520000000000000000000,9000000.000000000000000000,39,106
2025-11-16,147,42171738.810000000000000000,286882.5769387755102040816327,11000000.000000000000000000,36,68
2025-11-17,205,51444596.130000000000000000,250949.2494146341463414634146,10000000.000000000000000000,63,84
2025-11-18,407,452792605.529442000000000000,1112512.5442983832923832923833,200000000.000000000000000000,121,163
2025-11-19,272,26558133.607624000000000000,97640.1970868529411764705882,4238363.000000000000000000,88,115
2025-11-20,174,10459683.310000000000000000,60113.1224712643678160919540,600000.000000000000000000,50,75


In [0]:
%sql
SELECT
    SUM(transfer_count) AS gold_transfer_count
FROM workspace.token_risk_gold.daily_transfer_activity;

gold_transfer_count
2656


In [0]:
%sql
SELECT
    event_timestamp_utc,
    transaction_hash,
    from_address,
    to_address,
    token_amount
FROM workspace.token_risk_silver.sdb_transfers
ORDER BY token_amount DESC
LIMIT 20;

event_timestamp_utc,transaction_hash,from_address,to_address,token_amount
2025-11-18T06:07:01.000Z,0xaace63c263d97162ac59740bec34d587af733f926f8bea5ac217837d2a92e618,0x8e95308b721d3fa0066e3e02653d1709df4a8893,0x2e03695a934a3586b05fe2fba74543f7546b500d,200000000.000000000000000000
2025-11-18T06:18:11.000Z,0x05bddc2dec3af496ca4bf9591f4d86cb748b4eaf41bd2a9735284915cee23e05,0x2e03695a934a3586b05fe2fba74543f7546b500d,0x317c735bf712eb858446a13049f17fe492132101,200000000.000000000000000000
2025-11-14T09:38:00.000Z,0x2c9c9f07dafe8929f59cb4b09b0a36ec6e32762a543f80f37c894bdea9b597f1,0x8367e8e982b15b08314863b96cc08e3e73b221f3,0x51e3d44172868acc60d68ca99591ce4230bc75e0,20000000.000000000000000000
2025-11-13T19:47:40.000Z,0xb60b972b64790cf967d4101bb142ffc397e7be8670bd963c0af66f5db58d1f19,0x8e95308b721d3fa0066e3e02653d1709df4a8893,0x8367e8e982b15b08314863b96cc08e3e73b221f3,20000000.000000000000000000
2025-11-16T09:14:09.000Z,0xe3f9946384d86ec44f5edb3a121ae2d67650fa87a03a1c8a148c3f7993114975,0x2e03695a934a3586b05fe2fba74543f7546b500d,0x317c735bf712eb858446a13049f17fe492132101,11000000.000000000000000000
2025-11-16T09:08:41.000Z,0xd1f4893ba1d06087f2c4e04581ecc607e899cf4aac54d1181df81bc7eb1f15a1,0x8e95308b721d3fa0066e3e02653d1709df4a8893,0x2e03695a934a3586b05fe2fba74543f7546b500d,11000000.000000000000000000
2025-11-14T19:36:06.000Z,0x26b5e0fdf3f954ba0021faee414854349bda84a50b79d881d8201c35a250e3dd,0x8e95308b721d3fa0066e3e02653d1709df4a8893,0x2e03695a934a3586b05fe2fba74543f7546b500d,10000000.000000000000000000
2025-11-17T05:58:11.000Z,0x2265ca17faed511e2e50d04220574b18e64b73ae239573a98cb9829d055809e5,0x2e03695a934a3586b05fe2fba74543f7546b500d,0x317c735bf712eb858446a13049f17fe492132101,10000000.000000000000000000
2025-11-17T05:55:17.000Z,0x4a35f6c80eac1e5dc452fa89aa2b57066c272f4fc117d01fa8c8cfb06f9af888,0x8e95308b721d3fa0066e3e02653d1709df4a8893,0x2e03695a934a3586b05fe2fba74543f7546b500d,10000000.000000000000000000
2025-11-14T19:39:08.000Z,0x3c42b607f4c2d553bd345f9b78c19778107b7fb9628b96591109da93e2ed951e,0x2e03695a934a3586b05fe2fba74543f7546b500d,0x317c735bf712eb858446a13049f17fe492132101,10000000.000000000000000000


In [0]:
%sql
CREATE OR REPLACE TABLE
    workspace.token_risk_gold.wallet_flow_summary
USING DELTA
COMMENT 'Aggregated SDB sender-to-recipient flows during the listing research window'
AS
SELECT
    from_address,
    to_address,
    COUNT(*) AS transfer_count,
    COUNT(DISTINCT transaction_hash) AS transaction_count,
    SUM(token_amount) AS total_token_amount,
    AVG(token_amount) AS average_token_amount,
    MAX(token_amount) AS maximum_token_amount,
    MIN(event_timestamp_utc) AS first_transfer_utc,
    MAX(event_timestamp_utc) AS last_transfer_utc
FROM workspace.token_risk_silver.sdb_transfers
GROUP BY
    from_address,
    to_address;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT *
FROM workspace.token_risk_gold.wallet_flow_summary
ORDER BY total_token_amount DESC
LIMIT 20;

from_address,to_address,transfer_count,transaction_count,total_token_amount,average_token_amount,maximum_token_amount,first_transfer_utc,last_transfer_utc
0x8e95308b721d3fa0066e3e02653d1709df4a8893,0x2e03695a934a3586b05fe2fba74543f7546b500d,6,6,249000000.000000000000000000,41500000.0000000000000000000000,200000000.000000000000000000,2025-11-12T16:53:43.000Z,2025-11-18T06:07:01.000Z
0x2e03695a934a3586b05fe2fba74543f7546b500d,0x317c735bf712eb858446a13049f17fe492132101,6,6,249000000.000000000000000000,41500000.0000000000000000000000,200000000.000000000000000000,2025-11-13T17:21:08.000Z,2025-11-18T06:18:11.000Z
0x317c735bf712eb858446a13049f17fe492132101,0xa77cc444db535ea85eefcbac2dc6aa07cd445a70,926,926,109893339.980000000000000000,118675.3131533477321814254860,5328002.000000000000000000,2025-11-11T12:33:03.000Z,2025-11-20T19:58:23.000Z
0x8e95308b721d3fa0066e3e02653d1709df4a8893,0x8367e8e982b15b08314863b96cc08e3e73b221f3,1,1,20000000.000000000000000000,20000000.0000000000000000000000,20000000.000000000000000000,2025-11-13T19:47:40.000Z,2025-11-13T19:47:40.000Z
0x8367e8e982b15b08314863b96cc08e3e73b221f3,0x51e3d44172868acc60d68ca99591ce4230bc75e0,1,1,20000000.000000000000000000,20000000.0000000000000000000000,20000000.000000000000000000,2025-11-14T09:38:00.000Z,2025-11-14T09:38:00.000Z
0x899d608d224e6a9286cef42529c232b274616274,0x51e3d44172868acc60d68ca99591ce4230bc75e0,4,4,11226986.000000000000000000,2806746.5000000000000000000000,4227786.000000000000000000,2025-11-18T04:14:35.000Z,2025-11-19T18:21:33.000Z
0xa77cc444db535ea85eefcbac2dc6aa07cd445a70,0x899d608d224e6a9286cef42529c232b274616274,4,4,11226986.000000000000000000,2806746.5000000000000000000000,4227786.000000000000000000,2025-11-18T04:06:55.000Z,2025-11-19T18:17:17.000Z
0x1133209228922ed63279c309972cee654fb6d5fe,0x51e3d44172868acc60d68ca99591ce4230bc75e0,1,1,6000000.000000000000000000,6000000.0000000000000000000000,6000000.000000000000000000,2025-11-14T09:38:00.000Z,2025-11-14T09:38:00.000Z
0x8e95308b721d3fa0066e3e02653d1709df4a8893,0x1133209228922ed63279c309972cee654fb6d5fe,1,1,6000000.000000000000000000,6000000.0000000000000000000000,6000000.000000000000000000,2025-11-11T07:34:19.000Z,2025-11-11T07:34:19.000Z
0x4c79a027bcec17d30203a757eec863359310ce38,0x51e3d44172868acc60d68ca99591ce4230bc75e0,2,2,5665000.000000000000000000,2832500.0000000000000000000000,5169160.000000000000000000,2025-11-17T11:29:05.000Z,2025-11-18T18:04:53.000Z


In [0]:
%sql
SELECT
    percentile_approx(token_amount, 0.50) AS median_amount,
    percentile_approx(token_amount, 0.90) AS p90_amount,
    percentile_approx(token_amount, 0.95) AS p95_amount,
    percentile_approx(token_amount, 0.99) AS p99_amount,
    MAX(token_amount) AS maximum_amount
FROM workspace.token_risk_silver.sdb_transfers;

median_amount,p90_amount,p95_amount,p99_amount,maximum_amount
14414.000000000000000000,205762.000000000000000000,602457.000000000000000000,4227786.000000000000000000,200000000.000000000000000000


In [0]:
%sql
SELECT
    event_timestamp_utc,
    transaction_hash,
    from_address,
    to_address,
    token_amount,
    ROUND(
        (
            unix_timestamp(event_timestamp_utc)
            - unix_timestamp(TIMESTAMP '2025-11-14 09:30:00')
        ) / 60.0,
        2
    ) AS minutes_after_trading_open
FROM workspace.token_risk_silver.sdb_transfers
WHERE event_timestamp_utc >= TIMESTAMP '2025-11-14 09:30:00'
  AND event_timestamp_utc <  TIMESTAMP '2025-11-14 10:30:00'
ORDER BY token_amount DESC;

event_timestamp_utc,transaction_hash,from_address,to_address,token_amount,minutes_after_trading_open
2025-11-14T09:38:00.000Z,0x2c9c9f07dafe8929f59cb4b09b0a36ec6e32762a543f80f37c894bdea9b597f1,0x8367e8e982b15b08314863b96cc08e3e73b221f3,0x51e3d44172868acc60d68ca99591ce4230bc75e0,20000000.000000000000000000,8.00
2025-11-14T09:38:00.000Z,0x6294a907641ee93b786b54def53f68ae1b234e2040c126eed76253aafe9d647b,0x1133209228922ed63279c309972cee654fb6d5fe,0x51e3d44172868acc60d68ca99591ce4230bc75e0,6000000.000000000000000000,8.00
2025-11-14T09:38:00.000Z,0xa532c94cc107bceba78c5b5bbddce5771cf7e3f01d9c220c65280eb90aef56dc,0xe06bee4120f0fe4cb2fa9071234f061f603eaba1,0x51e3d44172868acc60d68ca99591ce4230bc75e0,5328002.000000000000000000,8.00
2025-11-14T09:36:04.000Z,0x876888b88f771765df1e75a995c63bfe1a7ea3887468219eff5d274727066ff2,0xb31b465107217afdcb371a1ae796138771076129,0x51e3d44172868acc60d68ca99591ce4230bc75e0,3711663.000000000000000000,6.07
2025-11-14T09:36:04.000Z,0x9e50d2c6870f4642f42abd8223289c90de4a7964a55722228bee2362d806fb51,0xe424c7096e1abfdbabd6303b09184281b13fe8ca,0x51e3d44172868acc60d68ca99591ce4230bc75e0,3114400.000000000000000000,6.07
2025-11-14T09:36:04.000Z,0x169572d52d5be81ac141369b0baebc346ce592e0f89b5a70293541294cee2ffd,0xe864db15812ff92c608ed5f8bb10afe6758ef510,0x51e3d44172868acc60d68ca99591ce4230bc75e0,3009200.000000000000000000,6.07
2025-11-14T10:25:30.000Z,0x12429958f7c7d669aa93875d60aaa29b95de5304fff5182a92d9bd616465232b,0x317c735bf712eb858446a13049f17fe492132101,0xa77cc444db535ea85eefcbac2dc6aa07cd445a70,2605515.000000000000000000,55.50
2025-11-14T10:25:30.000Z,0x12429958f7c7d669aa93875d60aaa29b95de5304fff5182a92d9bd616465232b,0xa77cc444db535ea85eefcbac2dc6aa07cd445a70,0xecdfb635b1fa1cfe3e34dfff204c412cf6bd4306,2594600.000000000000000000,55.50
2025-11-14T09:51:00.000Z,0x3741a96e118fed846e3dc8575859c93e1294c8d1577428a1c9b47a4d67d3d12b,0xde60a5f1d8eb15f26baf36727650cf90f1d1c81a,0x51e3d44172868acc60d68ca99591ce4230bc75e0,2500000.000000000000000000,21.00
2025-11-14T09:49:14.000Z,0x0722a4bbdf6d250143f026d6da21406f7ac3bd54583ce0fe8e0205d08243cf54,0xa77cc444db535ea85eefcbac2dc6aa07cd445a70,0xde60a5f1d8eb15f26baf36727650cf90f1d1c81a,2500000.000000000000000000,19.23


In [0]:
%sql
SELECT
    COUNT(*) AS transfer_count,
    SUM(token_amount) AS gross_token_volume,
    MAX(token_amount) AS maximum_transfer,
    COUNT(DISTINCT from_address) AS unique_senders,
    COUNT(DISTINCT to_address) AS unique_recipients
FROM workspace.token_risk_silver.sdb_transfers
WHERE event_timestamp_utc >= TIMESTAMP '2025-11-14 09:30:00'
  AND event_timestamp_utc <  TIMESTAMP '2025-11-14 10:30:00';

transfer_count,gross_token_volume,maximum_transfer,unique_senders,unique_recipients
284,114776276.300000000000000000,20000000.000000000000000000,139,70


In [0]:
%sql
WITH event_windows AS
(
    SELECT
        CASE
            WHEN event_timestamp_utc >= TIMESTAMP '2025-11-14 08:30:00'
             AND event_timestamp_utc <  TIMESTAMP '2025-11-14 09:30:00'
                THEN 'one_hour_before'

            WHEN event_timestamp_utc >= TIMESTAMP '2025-11-14 09:30:00'
             AND event_timestamp_utc <  TIMESTAMP '2025-11-14 10:30:00'
                THEN 'one_hour_after'
        END AS event_window,
        token_amount,
        from_address,
        to_address
    FROM workspace.token_risk_silver.sdb_transfers
    WHERE event_timestamp_utc >= TIMESTAMP '2025-11-14 08:30:00'
      AND event_timestamp_utc <  TIMESTAMP '2025-11-14 10:30:00'
)
SELECT
    event_window,
    COUNT(*) AS transfer_count,
    SUM(token_amount) AS gross_token_volume,
    AVG(token_amount) AS average_transfer,
    MAX(token_amount) AS maximum_transfer,
    COUNT(DISTINCT from_address) AS unique_senders,
    COUNT(DISTINCT to_address) AS unique_recipients
FROM event_windows
GROUP BY event_window
ORDER BY event_window;

event_window,transfer_count,gross_token_volume,average_transfer,maximum_transfer,unique_senders,unique_recipients
one_hour_after,284,114776276.300000000000000000,404141.8179577464788732394366,20000000.000000000000000000,139,70
one_hour_before,82,4428494.000000000000000000,54006.0243902439024390243902,515600.000000000000000000,2,40


In [0]:
%sql
SELECT
    to_address,
    COUNT(*) AS received_transfer_count,
    COUNT(DISTINCT from_address) AS unique_senders,
    SUM(token_amount) AS gross_received_amount,
    MAX(token_amount) AS maximum_received_transfer
FROM workspace.token_risk_silver.sdb_transfers
WHERE event_timestamp_utc >= TIMESTAMP '2025-11-14 09:30:00'
  AND event_timestamp_utc <  TIMESTAMP '2025-11-14 10:30:00'
GROUP BY to_address
ORDER BY gross_received_amount DESC
LIMIT 20;

to_address,received_transfer_count,unique_senders,gross_received_amount,maximum_received_transfer
0x51e3d44172868acc60d68ca99591ce4230bc75e0,141,136,78356892.900000000000000000,20000000.000000000000000000
0xa77cc444db535ea85eefcbac2dc6aa07cd445a70,65,1,18209391.700000000000000000,2605515.000000000000000000
0x5cf3f473772709bc4f395a60cf6f9a2bb997a26a,2,1,4879200.000000000000000000,2440000.000000000000000000
0xecdfb635b1fa1cfe3e34dfff204c412cf6bd4306,2,1,4594200.000000000000000000,2594600.000000000000000000
0xde60a5f1d8eb15f26baf36727650cf90f1d1c81a,2,1,3010000.000000000000000000,2500000.000000000000000000
0x9abb09b0d7b232975907a13582b41ce634da6835,1,1,1015600.000000000000000000,1015600.000000000000000000
0x7cf92c927d6486d38fc0bcc5e616e85841055745,1,1,999600.000000000000000000,999600.000000000000000000
0x103fa8a37e8e8e5cb5bd16ece980f31f95b37e85,1,1,499600.000000000000000000,499600.000000000000000000
0x99a9cad46665f50b394757924c5e677547506bc2,1,1,445400.000000000000000000,445400.000000000000000000
0x1eb2dc926e8d706196f3576a7348c7fbd4d6e2dc,2,1,329600.000000000000000000,299600.000000000000000000


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.token_risk_gold.large_transfers
USING DELTA
COMMENT 'SDB transfers at or above the listing-window P99 threshold'
AS

WITH threshold AS (
    SELECT
        percentile_approx(token_amount, 0.99) AS p99_amount
    FROM workspace.token_risk_silver.sdb_transfers
),

large_candidates AS (
    SELECT
        s.event_timestamp_utc,
        s.transaction_hash,
        s.block_number,
        s.transaction_index,
        s.from_address,
        s.to_address,
        s.token_amount,
        t.p99_amount,
        ROUND(
            s.token_amount / t.p99_amount,
            2
        ) AS p99_multiple
    FROM workspace.token_risk_silver.sdb_transfers AS s
    CROSS JOIN threshold AS t
    WHERE s.token_amount >= t.p99_amount
)

SELECT
    *,
    DENSE_RANK() OVER (
        ORDER BY token_amount DESC
    ) AS amount_rank
FROM large_candidates;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT *
FROM workspace.token_risk_gold.large_transfers
ORDER BY token_amount DESC;

event_timestamp_utc,transaction_hash,block_number,transaction_index,from_address,to_address,token_amount,p99_amount,p99_multiple,amount_rank
2025-11-18T06:07:01.000Z,0xaace63c263d97162ac59740bec34d587af733f926f8bea5ac217837d2a92e618,79170495,115,0x8e95308b721d3fa0066e3e02653d1709df4a8893,0x2e03695a934a3586b05fe2fba74543f7546b500d,200000000.000000000000000000,4227786.000000000000000000,47.31,1
2025-11-18T06:18:11.000Z,0x05bddc2dec3af496ca4bf9591f4d86cb748b4eaf41bd2a9735284915cee23e05,79170830,42,0x2e03695a934a3586b05fe2fba74543f7546b500d,0x317c735bf712eb858446a13049f17fe492132101,200000000.000000000000000000,4227786.000000000000000000,47.31,1
2025-11-13T19:47:40.000Z,0xb60b972b64790cf967d4101bb142ffc397e7be8670bd963c0af66f5db58d1f19,78979181,71,0x8e95308b721d3fa0066e3e02653d1709df4a8893,0x8367e8e982b15b08314863b96cc08e3e73b221f3,20000000.000000000000000000,4227786.000000000000000000,4.73,2
2025-11-14T09:38:00.000Z,0x2c9c9f07dafe8929f59cb4b09b0a36ec6e32762a543f80f37c894bdea9b597f1,79004091,101,0x8367e8e982b15b08314863b96cc08e3e73b221f3,0x51e3d44172868acc60d68ca99591ce4230bc75e0,20000000.000000000000000000,4227786.000000000000000000,4.73,2
2025-11-16T09:08:41.000Z,0xd1f4893ba1d06087f2c4e04581ecc607e899cf4aac54d1181df81bc7eb1f15a1,79089550,266,0x8e95308b721d3fa0066e3e02653d1709df4a8893,0x2e03695a934a3586b05fe2fba74543f7546b500d,11000000.000000000000000000,4227786.000000000000000000,2.60,3
2025-11-16T09:14:09.000Z,0xe3f9946384d86ec44f5edb3a121ae2d67650fa87a03a1c8a148c3f7993114975,79089714,109,0x2e03695a934a3586b05fe2fba74543f7546b500d,0x317c735bf712eb858446a13049f17fe492132101,11000000.000000000000000000,4227786.000000000000000000,2.60,3
2025-11-17T05:55:17.000Z,0x4a35f6c80eac1e5dc452fa89aa2b57066c272f4fc117d01fa8c8cfb06f9af888,79126943,129,0x8e95308b721d3fa0066e3e02653d1709df4a8893,0x2e03695a934a3586b05fe2fba74543f7546b500d,10000000.000000000000000000,4227786.000000000000000000,2.37,4
2025-11-17T05:58:11.000Z,0x2265ca17faed511e2e50d04220574b18e64b73ae239573a98cb9829d055809e5,79127030,35,0x2e03695a934a3586b05fe2fba74543f7546b500d,0x317c735bf712eb858446a13049f17fe492132101,10000000.000000000000000000,4227786.000000000000000000,2.37,4
2025-11-14T19:39:08.000Z,0x3c42b607f4c2d553bd345f9b78c19778107b7fb9628b96591109da93e2ed951e,79022125,17,0x2e03695a934a3586b05fe2fba74543f7546b500d,0x317c735bf712eb858446a13049f17fe492132101,10000000.000000000000000000,4227786.000000000000000000,2.37,4
2025-11-14T19:36:06.000Z,0x26b5e0fdf3f954ba0021faee414854349bda84a50b79d881d8201c35a250e3dd,79022034,109,0x8e95308b721d3fa0066e3e02653d1709df4a8893,0x2e03695a934a3586b05fe2fba74543f7546b500d,10000000.000000000000000000,4227786.000000000000000000,2.37,4


In [0]:
%sql
SELECT
    (SELECT COUNT(*)
     FROM workspace.token_risk_bronze.sdb_transfers
    ) AS bronze_records,

    (SELECT COUNT(*)
     FROM workspace.token_risk_silver.sdb_transfers
    ) AS silver_records,

    (SELECT SUM(transfer_count)
     FROM workspace.token_risk_gold.daily_transfer_activity
    ) AS daily_gold_records,

    (SELECT SUM(transfer_count)
     FROM workspace.token_risk_gold.wallet_flow_summary
    ) AS wallet_flow_records,

    (SELECT COUNT(*)
     FROM workspace.token_risk_gold.large_transfers
    ) AS large_transfer_records;

bronze_records,silver_records,daily_gold_records,wallet_flow_records,large_transfer_records
2656,2656,2656,2656,27
